In [1]:
import sys
print(sys.executable)

C:\Users\Usuario\anaconda3\envs\stockifai\python.exe


In [2]:
import pandas as pd

ruta = "../data/raw/Import-Movimientos.xlsx"
mov = pd.read_excel(ruta)
mov.shape


(70193, 8)

In [3]:
mov.head()

,KardexId,Repuesto,Descripcion,Fecha,Cantidad,Tipo,Documento,Deposito
0,4061504,16546ET00A,FILTRO DE AIRE,2025-07-24,1,E,VA-X-0001-00030325,CENTRAL
1,4061505,1520865F0E,VALVULA FILTRO ACEITE H4M KD,2025-07-24,1,E,VA-X-0001-00030325,CENTRAL
2,4061506,NIS7000008,LIMPIA PARABRISA X 1 LTS,2025-07-24,1,E,VA-X-0001-00030325,CENTRAL
3,4061507,NIS7123042,M-SUPER 2000 X3 10W40 24X1L,2025-07-24,4,E,VA-X-0001-00030325,D3
4,4061508,165463AW0J,FILTRO DE AIRE,2025-07-24,1,E,VA-X-0001-00030326,CENTRAL


In [4]:
mov.dtypes

KardexId                int64
Repuesto                  str
Descripcion               str
Fecha          datetime64[us]
Cantidad                int64
Tipo                      str
Documento                 str
Deposito                  str
dtype: object

In [16]:
mov.isna().sum()

KardexId       0
Repuesto       0
Descripcion    0
Fecha          0
Cantidad       0
Tipo           0
Documento      0
Deposito       0
dtype: int64

In [17]:
mov['Tipo'].value_counts()

Tipo
E    70193
Name: count, dtype: int64

In [18]:
mov['Deposito'].value_counts()

Deposito
CENTRAL      61975
D3            5999
PICKING       1976
PAJARERA       237
GARANTIAS        6
Name: count, dtype: int64

In [19]:
mov['Documento']

0        VA-X-0001-00030325
1        VA-X-0001-00030325
2        VA-X-0001-00030325
3        VA-X-0001-00030325
4        VA-X-0001-00030326
                ...        
70188    VA-X-0001-00015882
70189    VA-X-0001-00015882
70190    VA-X-0001-00015882
70191    VA-X-0001-00015882
70192    VA-X-0001-00015882
Name: Documento, Length: 70193, dtype: str

In [20]:
mov["prefijo"] = mov["Documento"].str.split("-").str[0]
mov["prefijo"].value_counts()

prefijo
VA    48298
FC    21895
Name: count, dtype: int64

In [21]:
mov.head()

,KardexId,Repuesto,Descripcion,Fecha,Cantidad,Tipo,Documento,Deposito,prefijo
0,4061504,16546ET00A,FILTRO DE AIRE,2025-07-24,1,E,VA-X-0001-00030325,CENTRAL,VA
1,4061505,1520865F0E,VALVULA FILTRO ACEITE H4M KD,2025-07-24,1,E,VA-X-0001-00030325,CENTRAL,VA
2,4061506,NIS7000008,LIMPIA PARABRISA X 1 LTS,2025-07-24,1,E,VA-X-0001-00030325,CENTRAL,VA
3,4061507,NIS7123042,M-SUPER 2000 X3 10W40 24X1L,2025-07-24,4,E,VA-X-0001-00030325,D3,VA
4,4061508,165463AW0J,FILTRO DE AIRE,2025-07-24,1,E,VA-X-0001-00030326,CENTRAL,VA


In [23]:
pd.crosstab(mov['prefijo'], mov['Deposito'])

Deposito,CENTRAL,D3,GARANTIAS,PAJARERA,PICKING
prefijo,,,,,
FC,19902,685,1,83,1224
VA,42073,5314,5,154,752


In [24]:
mov.groupby("Deposito")["Fecha"].agg(["min","max"])

,min,max
Deposito,,
CENTRAL,2020-08-12,2025-07-24
D3,2020-08-12,2025-07-24
GARANTIAS,2021-02-01,2024-10-23
PAJARERA,2024-06-28,2025-07-21
PICKING,2020-08-12,2025-07-23


In [25]:
mov["semana"] = mov['Fecha'].dt.to_period('W').dt.start_time

In [26]:
mov[["Repuesto", "Fecha", "semana"]].head(10)

,Repuesto,Fecha,semana
0,16546ET00A,2025-07-24,2025-07-21
1,1520865F0E,2025-07-24,2025-07-21
2,NIS7000008,2025-07-24,2025-07-21
3,NIS7123042,2025-07-24,2025-07-21
4,165463AW0J,2025-07-24,2025-07-21
5,1520865F0E,2025-07-24,2025-07-21
6,NIS7000008,2025-07-24,2025-07-21
7,NIS7123042,2025-07-24,2025-07-21
8,165463AW0J,2025-07-24,2025-07-21
9,1520865F0E,2025-07-24,2025-07-21


In [27]:
semanal = mov.groupby(['Repuesto','semana'], as_index=False)['Cantidad'].sum()

In [28]:
semanal.shape

(28116, 3)

In [29]:
por_repuesto = semanal.groupby('Repuesto')['semana'].count()

In [34]:
por_repuesto.sort_values(ascending=False).head(10)


Repuesto
16546ED00J    259
NIS7000008    259
1520865F0E    259
165463AW0J    259
165465RA0A    258
LAMPARAPOS    258
NIS0000166    257
NIS7122058    251
27891EL00B    238
272773HU0B    235
Name: semana, dtype: int64

In [35]:
por_repuesto.describe()

count    3901.000000
mean        7.207383
std        21.909787
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max       259.000000
Name: semana, dtype: float64

In [37]:
for minimo in [2,5,10,26,52]:
    cuantos = (por_repuesto>=minimo).sum()
    print(f"Venden en al menos {minimo} semanas: {cuantos} repuestos ")

Venden en al menos 2 semanas: 2084 repuestos 
Venden en al menos 5 semanas: 904 repuestos 
Venden en al menos 10 semanas: 494 repuestos 
Venden en al menos 26 semanas: 225 repuestos 
Venden en al menos 52 semanas: 99 repuestos 


In [38]:
unidades = semanal.groupby('Repuesto')['Cantidad'].sum()

In [39]:
total_unidades = unidades.sum()

In [40]:
total_repuestos = len(por_repuesto)

In [41]:
filas = []

In [42]:
for minimo in [1,2,5,10,26,52]:
    pasa = por_repuesto >= minimo
    filas.append({
        "minimo_semanas" : minimo,
        "repuestos" : pasa.sum(),
        "pct_repuestos" : round(pasa.sum() / total_repuestos * 100,1),
        "pct_unidades" : round(unidades[pasa].sum() / total_unidades * 100,1)
    })

In [43]:
tabla_cortes = pd.DataFrame(filas)
tabla_cortes

,minimo_semanas,repuestos,pct_repuestos,pct_unidades
0,1,3901,100.0,100.0
1,2,2084,53.4,97.9
2,5,904,23.2,94.2
3,10,494,12.7,90.1
4,26,225,5.8,85.5
5,52,99,2.5,79.2


### Decisión: universo de repuestos
- Entran al modelo los repuestos con al menos 10 semanas con venta:
  494 repuestos (13% del catálogo), 90% de las unidades vendidas.
- Se evaluará el error por separado para los repuestos con 10-25 semanas
  y con 26 o más. Si el modelo no le gana a la regla simple en el grupo 10-25,
  el corte sube a 26 (225 repuestos, 86% de las unidades).
- El resto de los repuestos recibe una regla simple (baseline).